In [ ]:
# TF-IDF + BiLSTM Hybrid
import json, pandas as pd, numpy as np, torch, torch.nn as nn, re
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

# Load data
with open('/kaggle/input/fetched-data-final-2/fetched_data_final_dedup.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)
df['label'] = df['label'].astype(int)
X, y = df['text'], df['label']

# Build vocabulary
def tokenize(text): return re.findall(r'\b\w+\b', text.lower())
all_tokens = [token for text in X for token in tokenize(text)]
vocab = {word: i+2 for i, (word, _) in enumerate(Counter(all_tokens).most_common(10000))}
vocab.update({'<PAD>': 0, '<UNK>': 1})

def text_to_sequence(text, max_len=100):
    tokens = tokenize(text)
    return [vocab.get(token, 1) for token in tokens][:max_len]

sequences = [text_to_sequence(text) for text in X]

# TF-IDF features
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_tfidf = StandardScaler().fit_transform(tfidf.fit_transform(X).toarray())

# Split data
X_seq_train, X_seq_val, X_tfidf_train, X_tfidf_val, y_train, y_val = train_test_split(
    sequences, X_tfidf, y, test_size=0.2, random_state=42, stratify=y)

class HybridDataset(torch.utils.data.Dataset):
    def __init__(self, sequences, tfidf_features, labels):
        self.sequences, self.tfidf_features, self.labels = sequences, tfidf_features, labels
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx):
        return (torch.tensor(self.sequences[idx], dtype=torch.long),
                torch.tensor(self.tfidf_features[idx], dtype=torch.float32),
                torch.tensor(self.labels.iloc[idx], dtype=torch.float32))

def collate_fn(batch):
    sequences, tfidf_features, labels = zip(*batch)
    return (pad_sequence(sequences, batch_first=True, padding_value=0),
            torch.stack(tfidf_features), torch.stack(labels))

class HybridTfidfBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=64, tfidf_dim=1000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2 + tfidf_dim, 64)
        self.classifier = nn.Linear(64, 1)
        
    def forward(self, sequences, tfidf_features):
        embedded = self.embedding(sequences)
        _, (h_n, _) = self.lstm(embedded)
        lstm_features = self.dropout(torch.cat((h_n[-2], h_n[-1]), dim=1))
        combined = torch.cat([lstm_features, tfidf_features], dim=1)
        x = self.dropout(torch.relu(self.fc(combined)))
        return torch.sigmoid(self.classifier(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = DataLoader(HybridDataset(X_seq_train, X_tfidf_train, y_train), 
                          batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(HybridDataset(X_seq_val, X_tfidf_val, y_val), 
                        batch_size=32, collate_fn=collate_fn)

model = HybridTfidfBiLSTM(len(vocab)).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(20):
    model.train()
    train_loss = 0
    for sequences, tfidf_features, labels in train_loader:
        sequences, tfidf_features, labels = sequences.to(device), tfidf_features.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(sequences, tfidf_features).squeeze(), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for sequences, tfidf_features, labels in val_loader:
            sequences, tfidf_features, labels = sequences.to(device), tfidf_features.to(device), labels.to(device)
            preds = model(sequences, tfidf_features).squeeze()
            val_loss += nn.functional.binary_cross_entropy(preds, labels).item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_preds = (np.array(all_preds) > 0.5).astype(int)
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    print(f'Epoch [{epoch+1}/20] | Train Loss: {train_loss:.4f} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}')

In [ ]:
# Visualization
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Final evaluation
model.eval()
final_preds, final_labels = [], []
with torch.no_grad():
    for sequences, tfidf_features, labels in val_loader:
        preds = model(sequences.to(device), tfidf_features.to(device)).squeeze().cpu().numpy()
        final_preds.extend(preds)
        final_labels.extend(labels.numpy())

final_preds_binary = (np.array(final_preds) > 0.5).astype(int)

# Plots
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
cm = confusion_matrix(final_labels, final_preds_binary)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.subplot(1, 3, 2)
plt.hist(final_preds, bins=50, alpha=0.7, edgecolor='black')
plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold')
plt.title('Prediction Distribution')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 3, 3)
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
values = [accuracy_score(final_labels, final_preds_binary),
          precision_score(final_labels, final_preds_binary, zero_division=0),
          recall_score(final_labels, final_preds_binary, zero_division=0),
          f1_score(final_labels, final_preds_binary, zero_division=0)]
plt.bar(metrics, values, color=['skyblue', 'lightgreen', 'salmon', 'gold'])
plt.title('Metrics')
plt.ylabel('Score')
plt.ylim(0, 1)
for i, v in enumerate(values):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(final_labels, final_preds_binary, target_names=['Non-Hate', 'Hate']))
print(f'\nGPU Used: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
